In [ ]:
import duckdb
import pandas as pd

In [9]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)


In [20]:
df = con.execute("""
                 SELECT * FROM(
                 SELECT * , ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                 FROM bronze_produtos 
                 WHERE data_ingestao >= '2025-01-11')
                 WHERE row = 1""").fetchdf()
df.head(10)


,NATBR,MAKTX,WERKS,MAINST,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2025-04-18 23:40:36.493350,1
1,10002,MARTELO,BT20,200,200,z0019_1.csv,2025-04-18 23:40:36.493350,1
2,10005,MACHADO,BT50,500,500,z0019_2.csv,2025-04-18 23:51:57.787581,1
3,10003,PREGO,BT10,100,100,z0019_2.csv,2025-04-18 23:51:57.787581,1
4,10006,SERRA,BT60,600,600,z0019_2.csv,2025-04-18 23:51:57.787581,1


In [22]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])    
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nm_produto"})
df_final = df_final.rename(columns={"WERKS":"id_categoria"})
df_final = df_final.rename(columns={"MAINST":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_preco"})

df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT20,200,200
2,10005,MACHADO,BT50,500,500
3,10003,PREGO,BT10,100,100
4,10006,SERRA,BT60,600,600


In [ ]:
df2 = df_final
df2 = df2.astype(
    {
        'id': int,
        'nm_produto': str,
        'id_categoria': str,
        'id_fornecedor': int,
        'vl_preco': float
    }
)
df2.dtypes
df2.head(10)

id                 int32
nm_produto        object
id_categoria      object
id_fornecedor      int32
vl_preco         float64
dtype: object

In [30]:
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
        id BIGINT,
        nm_produto TEXT,
        id_categoria TEXT,
        id_fornecedor BIGINT,
        vl_preco FLOAT
    )
""")

In [31]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT20,200,200.0
2,10005,MACHADO,BT50,500,500.0
3,10003,PREGO,BT10,100,100.0
4,10006,SERRA,BT60,600,600.0


In [33]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [34]:
df_resultado = con.execute("""select * from produtos""").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT20,200,200.0
2,10005,MACHADO,BT50,500,500.0
3,10003,PREGO,BT10,100,100.0
4,10006,SERRA,BT60,600,600.0


In [35]:
con.close()